# Predição de Gênero - Análise de Reviews

- **Aluno:** Pedro Yutaro Mont Morency Nakamura  
- **Data:** 17/05/2026
- **Objetivo:** Desenvolver um modelo de classificação para predizer gênero...

## 0. Configurações e Importações de Dependẽncias e Dados

In [1]:
import os
import pickle
import pandas as pd
import numpy as np

# Visualização de Dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-Processamento
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
from tqdm import tqdm
from spacy.matcher import Matcher

# Estruturas e estatística
from scipy.stats import loguniform, randint, uniform

# Classificadores e seleção
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler

# Validação e busca de hiperparâmetros
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

### 0.1 Configuração de Gráficos

In [5]:
# Configurações de visualização
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

### 0.2 Carregamento do modelo SpaCy para Português

In [6]:
try:
    # Carrega apenas componentes necessarios para POS/lemmas.
    nlp = spacy.load("pt_core_news_lg", disable=["parser", "ner"])
    print("✅ SpaCy pt_core_news_lg carregado com sucesso!")
except Exception as e:
    print(f"❌ Erro ao carregar o modelo: {e}")

✅ SpaCy pt_core_news_lg carregado com sucesso!


### 0.3 Importação dos Datasets

In [7]:
# Caminhos para os dados
TRAIN_PATH = './data/treino.csv'
VALIDATION_PATH = './data/validacao.csv'

In [8]:
# Carregando CSVs
df_train = pd.read_csv(TRAIN_PATH, encoding='utf-8', low_memory=False)
df_val = pd.read_csv(VALIDATION_PATH, encoding='utf-8', low_memory=False)

## 1. Exploração dos Dados

In [9]:
print(f"Treino: {len(df_train)} reviews")
print(f"Distribuição de gênero:\n{df_train['reviewer_gender'].value_counts()}")
df_train.head()

Treino: 76942 reviews
Distribuição de gênero:
reviewer_gender
M    39699
F    37243
Name: count, dtype: int64


,submission_date,reviewer_id,product_id,product_name,product_brand,site_category_lv1,site_category_lv2,review_title,overall_rating,recommend_to_a_friend,review_text,reviewer_birth_year,reviewer_gender,reviewer_state
0,2018-01-25 15:09:27,a06d0380bcd447431774739131312d3532e1018988da8c...,129610281,"iPhone 7 Plus 128GB dourado Tela Retina HD 5,5...",apple,Celulares e Smartphones,Smartphone,Iphone,3,Yes,É Verdade Mesmo Gente Essa Promocao? To queren...,1994.0,F,BA
1,2018-05-14 16:20:25,62bbbab0822b6917233e8a9846dc3e7acbdfe6b4df788a...,120934895,"TV LED 32"" Sony KDL-32R305B HD com Conversor D...",sony,TV e Home Theater,TV,Marca conceituada,3,Yes,"Acredito ser uma Boa TV, não tenho como ava...",1965.0,M,BA
2,2018-05-19 15:43:40,27908d4eea73f655ca4d50adc1c373efccc3b748d4a779...,15844032,Suporte P/ Notebook C/ Cooler Gamer Warrior - ...,NaN,Informática e Acessórios,Peças para Notebook,Excelente aquisição,5,Yes,A compra do produto confirmou a qualidade de o...,1951.0,M,PE
3,2018-01-14 14:35:10,b1ddb4c179231ffb650097bec1ad8033dc8c2bad60a846...,132444050,Smartphone Motorola Moto G 5S Dual Chip Androi...,NaN,Celulares e Smartphones,Smartphone,Atende as necessidades,4,Yes,Celular intermediário de bom custo benefício. ...,1978.0,F,PA
4,2018-03-08 12:40:03,4188541be9484e0e49c4803e767354c3e84e419a2d23da...,132522724,Panela de Pressão Elétrica Digital Philco Pppv...,NaN,Eletroportáteis,Panela Elétrica,Muito bom,4,Yes,"Prática, fácil de limpar, excelente produto. R...",1980.0,F,RJ


In [10]:
print(f"Validação: {len(df_val)} reviews")
print(f"Distribuição de gênero:\n{df_val['reviewer_gender'].value_counts()}")
df_val.head()

Validação: 25647 reviews
Distribuição de gênero:
reviewer_gender
M    13233
F    12414
Name: count, dtype: int64


,submission_date,reviewer_id,product_id,product_name,product_brand,site_category_lv1,site_category_lv2,review_title,overall_rating,recommend_to_a_friend,review_text,reviewer_birth_year,reviewer_gender,reviewer_state
0,2018-04-30 19:24:28,fabc81c4fa9efa6867a78d82ce2713099ed1eea5df9d80...,31575567,Vela Led Decorativa 15cm Pilha Amarelo-Der/07-...,NaN,Decoração,Luminária,Efeito bonito,4,Yes,Gostei do efeito. Parece uma vela de verdade. ...,1954.0,F,RS
1,2018-04-27 17:48:48,6fbac43a75fd2dbb7e74687472779b74cc9602a566be74...,132474081,Smartphone Moto G 5S Dual Chip Android 7.0 Tel...,NaN,Celulares e Smartphones,Smartphone,"Hardware fraco, mas conjunto excelente.",4,Yes,Não me arrependo de ter comprado o G5S à conco...,1996.0,M,RJ
2,2018-03-06 18:45:17,e85bcc86d48fdd1462bada4d5f4f19cde9f86acf157992...,16215756,Cadeira Amanda Medalhao Marrom Rivatti,NaN,Móveis,Cadeira,Excelente qualidade,5,Yes,"Produto recebido dentro do prazo, acondicionad...",1973.0,F,SP
3,2018-05-18 07:00:58,5e054a67eb8f4ab98b0e53c532843f65a49ed17aef8214...,12298660,Kit 2 peças Holofote Refletor Super Led Duplo ...,NaN,Casa e Construção,Iluminação,Produto de Ótima qualidade,4,Yes,"Produto de Ótima qualidade, atendeu todas as m...",1969.0,M,SP
4,2018-04-21 06:31:51,9b22f24daed6d47b570be15dc928f8d522e8e1315f7d0c...,132537782,Smartphone Samsung Galaxy J7 Pro Android 7.0 T...,NaN,Celulares e Smartphones,Smartphone,Satisfeito,5,Yes,"Muito bom aparelho. Fotos e vídeos perfeitos, ...",1973.0,M,SP


### 3.1 Contagem de valores e tipos

In [ ]:
print('\n<===== TRAIN DATAFRAME =====>')
df_train.info()


<===== TRAIN DATAFRAME =====>
<class 'pandas.DataFrame'>
RangeIndex: 76942 entries, 0 to 76941
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   submission_date        76942 non-null  str    
 1   reviewer_id            76942 non-null  str    
 2   product_id             76942 non-null  str    
 3   product_name           76903 non-null  str    
 4   product_brand          24162 non-null  str    
 5   site_category_lv1      76942 non-null  str    
 6   site_category_lv2      74687 non-null  str    
 7   review_title           76776 non-null  str    
 8   overall_rating         76942 non-null  int64  
 9   recommend_to_a_friend  76932 non-null  str    
 10  review_text            75056 non-null  str    
 11  reviewer_birth_year    75753 non-null  float64
 12  reviewer_gender        76942 non-null  str    
 13  reviewer_state         76942 non-null  str    
dtypes: float64(1), int64(1), str(12)
m

In [12]:
print('\n<===== VALIDATION DATAFRAME =====>')
df_val.info()


<===== VALIDATION DATAFRAME =====>
<class 'pandas.DataFrame'>
RangeIndex: 25647 entries, 0 to 25646
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   submission_date        25647 non-null  str    
 1   reviewer_id            25647 non-null  str    
 2   product_id             25647 non-null  int64  
 3   product_name           25629 non-null  str    
 4   product_brand          8048 non-null   str    
 5   site_category_lv1      25645 non-null  str    
 6   site_category_lv2      24875 non-null  str    
 7   review_title           25598 non-null  str    
 8   overall_rating         25647 non-null  int64  
 9   recommend_to_a_friend  25646 non-null  str    
 10  review_text            25044 non-null  str    
 11  reviewer_birth_year    25242 non-null  float64
 12  reviewer_gender        25647 non-null  str    
 13  reviewer_state         25647 non-null  str    
dtypes: float64(1), int64(2), str(

#### Verificando Valores Nulos

In [13]:
df_train.isna().sum()

submission_date              0
reviewer_id                  0
product_id                   0
product_name                39
product_brand            52780
site_category_lv1            0
site_category_lv2         2255
review_title               166
overall_rating               0
recommend_to_a_friend       10
review_text               1886
reviewer_birth_year       1189
reviewer_gender              0
reviewer_state               0
dtype: int64

In [14]:
df_val.isna().sum()

submission_date              0
reviewer_id                  0
product_id                   0
product_name                18
product_brand            17599
site_category_lv1            2
site_category_lv2          772
review_title                49
overall_rating               0
recommend_to_a_friend        1
review_text                603
reviewer_birth_year        405
reviewer_gender              0
reviewer_state               0
dtype: int64

## 3. Pré-Processamento

blablabla

In [ ]:
# Limpeza, normalização, etc.
def preprocessar(texto: str):
    # Seu código aqui
    texto_processado = texto.lower()
    return texto_processado

X_train_clean = df_train['review_text'].apply(preprocessar)

## 4. Representação Textual

In [ ]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train_clean)

## 5. Treinamento

In [ ]:
model = LogisticRegression()
model.fit(X_train_vec, y_train)

## 6. Validação e Métricas

In [ ]:
y_pred = model.predict(X_val_vec)
f1 = f1_score(y_val, y_pred, average='weighted')
print(f"F1 Score: {f1:.4f}")

## 7. Salvar Modelo

In [ ]:
def save_model(model, filename: str, path = './output/'):
  localefile = path + filename + '.pkl'
  pickle.dump(model, open(localefile, 'wb'))
  print("✓ Modelo salvo!")

## 8. Conclusões

- Melhor F1 Score alcançado: 0.XXXX
- Principais desafios: ...
- Possíveis melhorias: ...